In [1]:
import glob
from pathlib import Path
import numpy as np
import polars as pl
import scipy.stats as stats

try:
    from statsmodels.tsa.stattools import adfuller
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False

DATA_DIR = Path("take-home-project/data/orderbook")
files = sorted(DATA_DIR.glob("*.parquet"))
print(f"Found {len(files)} orderbook files: {[f.name for f in files]}")

Found 3 orderbook files: ['2026-03-19.parquet', '2026-03-20.parquet', '2026-03-21.parquet']


In [2]:
# Load orderbook snapshots and compute mid-price, bid-ask spread, and order flow imbalance
df_list = [pl.scan_parquet(f).select(["datetime", "bid_price_1", "ask_price_1", "bid_qty_1", "ask_qty_1"]) for f in files]
df = pl.concat(df_list).collect().sort("datetime")

df = df.with_columns([
    ((pl.col("bid_price_1") + pl.col("ask_price_1")) / 2.0).alias("mid_price"),
    (pl.col("ask_price_1") - pl.col("bid_price_1")).alias("spread"),
    ((pl.col("bid_qty_1") - pl.col("ask_qty_1")) / (pl.col("bid_qty_1") + pl.col("ask_qty_1"))).alias("imbalance")
])

# Resample at 1-second intervals
df_1s = (
    df.group_by_dynamic("datetime", every="1s")
    .agg([pl.col("mid_price").last(), pl.col("spread").last(), pl.col("imbalance").last()])
    .drop_nulls()
)

print(f"Total records loaded: {len(df):,}")
print(f"Total 1-second resampled rows: {len(df_1s):,}")

Total records loaded: 3,581,577
Total 1-second resampled rows: 258,932


In [ ]:
def calibrate_ou_and_adf(series: np.ndarray, dt: float = 1.0, series_name: str = "Series"):
    """
    Perform Dickey-Fuller / ADF test and fit Ornstein-Uhlenbeck parameters.
    """
    y = np.diff(series)
    x = series[:-1]
    
    b, a, r_value, p_value, std_err = stats.linregress(x, y)
    df_t_stat = b / std_err
    
    theta = -b / dt
    mu = -a / b if b != 0 else np.nan
    half_life = np.log(2) / theta if theta > 0 else np.inf
    
    print(f"=== {series_name} Analysis (dt = {dt}s) ===")
    print(f"  Linear Regression Slope (b): {b:.8f}")
    print(f"  Dickey-Fuller t-statistic:   {df_t_stat:.4f}")
    adf_res = adfuller(series, autolag='AIC')
    print(f"  ADF Test Statistic:          {adf_res[0]:.4f}")
    print(f"  ADF p-value:                 {adf_res[1]:.6f}")
    print(f"  Mean Reversion Speed (theta):{theta:.8f} s^-1")
    print(f"  Long-Term Mean (mu):         {mu:.6f}")
    if np.isfinite(half_life):
        print(f"  Half-Life:                   {half_life:.2f} seconds ({half_life/60:.2f} mins / {half_life/3600:.2f} hours)")
    else:
        print(f"  Half-Life:                   Infinite (Non mean-reverting / Divergent)")
    print()
    return {"b": b, "t_stat": df_t_stat, "theta": theta, "mu": mu, "half_life": half_life}

In [4]:
# Test Asset Price (Mid-Price) Mean Reversion
res_mid = calibrate_ou_and_adf(df_1s["mid_price"].to_numpy(), dt=1.0, series_name="ETH Perpetual Mid-Price")

=== ETH Perpetual Mid-Price Analysis (dt = 1.0s) ===
  Linear Regression Slope (b): -0.00003153
  Dickey-Fuller t-statistic:   -1.7980
  Approx 5% Critical Value:    ~ -2.86
  Mean Reversion Speed (theta):0.00003153 s^-1
  Long-Term Mean (mu):         2137.131590
  Half-Life:                   21981.78 seconds (366.36 mins / 6.11 hours)


In [5]:
# Test Market Microstructure Features (Spread & Order Flow Imbalance)
res_sp = calibrate_ou_and_adf(df_1s["spread"].to_numpy(), dt=1.0, series_name="Bid-Ask Spread")
res_imb = calibrate_ou_and_adf(df_1s["imbalance"].to_numpy(), dt=1.0, series_name="Order Book Imbalance (OFI)")

=== Bid-Ask Spread Analysis (dt = 1.0s) ===
  Linear Regression Slope (b): -1.00055314
  Dickey-Fuller t-statistic:   -509.1322
  Approx 5% Critical Value:    ~ -2.86
  Mean Reversion Speed (theta):1.00055314 s^-1
  Long-Term Mean (mu):         0.100176
  Half-Life:                   0.69 seconds (0.01 mins / 0.00 hours)

=== Order Book Imbalance (OFI) Analysis (dt = 1.0s) ===
  Linear Regression Slope (b): -0.25490270
  Dickey-Fuller t-statistic:   -194.4771
  Approx 5% Critical Value:    ~ -2.86
  Mean Reversion Speed (theta):0.25490270 s^-1
  Long-Term Mean (mu):         0.007433
  Half-Life:                   2.72 seconds (0.05 mins / 0.00 hours)


## Conclusions & Market Making Takeaways

### 1. Underlying Asset Level (Mid-Price)
- **ADF / Dickey-Fuller Result**: The t-statistic (`-1.80`) is above the 5% critical value (`~ -2.86`). We **cannot reject the null hypothesis of a unit root**.
- **OU Half-Life**: Estimated at **~21,982 seconds (~6.1 hours)**.
- **Implication**: Over intraday trading windows, the ETH perpetual price behaves largely like a random walk (non-stationary). Pure macro directional mean-reversion strategies on the price level alone are ineffective.

### 2. Microstructural Features (Spread & Imbalance)
- **ADF / Dickey-Fuller Result**: Both spread ($t \approx -509$) and order flow imbalance ($t \approx -194$) exhibit extreme statistical significance ($p < 0.0001$), confirming strong stationarity.
- **OU Half-Life**:
  - **Bid-Ask Spread**: Half-life $\approx \mathbf{0.69\text{ seconds}}$. Shocks to the spread (such as liquidity sweeps) mean-revert almost immediately.
  - **Order Book Imbalance**: Half-life $\approx \mathbf{2.72\text{ seconds}}$. Order flow skew mean-reverts within a few seconds.
- **Implication for Market Making Bot**: Instead of betting on price reversion, our market maker should:
  1. Quote aggressively when spreads widen beyond their equilibrium ($0.69\text{s}$ decay).
  2. Skew quotes (adjust reservation price) dynamically based on short-term order book imbalances ($2.72\text{s}$ decay).